# DL-02 Regularization Effects

- Module: 07 Deep Learning Systems
- Topic: comparing dropout and weight decay on an intentionally over-parameterized MLP
- Estimated runtime: 6 to 10 minutes once `torch` and `matplotlib` are installed
- Prerequisites: empirical risk minimization, train/validation splits, and Module 01 optimization labs
- Expected outputs: train/validation curves, generalization-gap comparisons, and a short Monte Carlo dropout demo


## Why this notebook exists

This notebook makes overfitting easy to observe on purpose.
We train a wide MLP on a small nonlinear dataset and compare:

- no explicit regularization;
- weight decay only;
- dropout only;
- dropout plus weight decay.

The point is not to prove a universal ranking.
The point is to see that these methods regularize in different ways.


In [ ]:
from __future__ import annotations

import random

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 11
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")


## Theory checkpoint

Weight decay adds a penalty

$$
\frac{\lambda}{2}\lVert \theta \rVert_2^2
$$

to the objective, which induces multiplicative shrinkage under gradient descent.

Dropout instead samples masks and trains many thinned subnetworks that share parameters.
That gives two complementary interpretations:

- a stochastic regularizer that discourages fragile co-adaptation;
- an approximate ensemble over subnetworks.


In [ ]:
def make_small_dataset(n_samples: int = 420):
    x = torch.empty(n_samples, 2).uniform_(-2.0, 2.0)
    y = ((x[:, 0] * x[:, 1]) + 0.35 * torch.sin(3 * x[:, 0]) > 0).long()
    x = x + 0.2 * torch.randn_like(x)
    return x, y


x, y = make_small_dataset()
permutation = torch.randperm(len(x))
train_idx = permutation[:140]
val_idx = permutation[140:280]
test_idx = permutation[280:]

train_dataset = TensorDataset(x[train_idx], y[train_idx])
val_dataset = TensorDataset(x[val_idx], y[val_idx])
test_dataset = TensorDataset(x[test_idx], y[test_idx])


class RegularizedMLP(nn.Module):
    def __init__(self, dropout_p: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 128),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(128, 2),
        )

    def forward(self, x):
        return self.net(x)


def run_experiment(dropout_p: float, weight_decay: float, epochs: int = 120):
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=128)
    test_loader = DataLoader(test_dataset, batch_size=128)

    model = RegularizedMLP(dropout_p=dropout_p).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for _ in range(epochs):
        model.train()
        total_loss = 0.0
        total_count = 0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(xb)
            total_count += len(xb)
        history["train_loss"].append(total_loss / total_count)

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_count = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                logits = model(xb.to(device))
                loss = loss_fn(logits, yb.to(device))
                val_loss += loss.item() * len(xb)
                val_correct += (logits.argmax(dim=1) == yb.to(device)).sum().item()
                val_count += len(xb)
        history["val_loss"].append(val_loss / val_count)
        history["val_acc"].append(val_correct / val_count)

    model.eval()
    test_correct = 0
    test_count = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            logits = model(xb.to(device))
            test_correct += (logits.argmax(dim=1) == yb.to(device)).sum().item()
            test_count += len(xb)
    history["test_acc"] = test_correct / test_count
    return model, history


In [ ]:
settings = {
    "baseline": {"dropout_p": 0.0, "weight_decay": 0.0},
    "weight_decay": {"dropout_p": 0.0, "weight_decay": 1e-3},
    "dropout": {"dropout_p": 0.35, "weight_decay": 0.0},
    "both": {"dropout_p": 0.35, "weight_decay": 1e-3},
}

trained_models = {}
histories = {}
for name, kwargs in settings.items():
    model, history = run_experiment(**kwargs)
    trained_models[name] = model
    histories[name] = history

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for name, history in histories.items():
    axes[0].plot(history["train_loss"], linestyle="--", label=f"{name} train")
    axes[0].plot(history["val_loss"], label=f"{name} val")
    axes[1].plot(history["val_acc"], label=name)

axes[0].set_title("Loss curves")
axes[1].set_title("Validation accuracy")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

{name: round(history["test_acc"], 3) for name, history in histories.items()}


In [ ]:
def mc_dropout_probability(model: nn.Module, inputs: torch.Tensor, samples: int = 40):
    model.train()
    draws = []
    with torch.no_grad():
        for _ in range(samples):
            probs = torch.softmax(model(inputs.to(device)), dim=1)[:, 1]
            draws.append(probs.cpu())
    stacked = torch.stack(draws)
    return stacked.mean(dim=0), stacked.std(dim=0)


probe_x = x[test_idx[:12]]
mean_prob, std_prob = mc_dropout_probability(trained_models["dropout"], probe_x)
list(zip(mean_prob[:5].tolist(), std_prob[:5].tolist()))


## Interpretation

Typical observations:

- the baseline model drives training loss very low and then opens a large validation gap;
- weight decay often smooths that gap by shrinking parameters;
- dropout usually slows optimization but can improve validation accuracy by reducing reliance on any one hidden path;
- dropout plus weight decay can help, but too much combined regularization can underfit.

The Monte Carlo dropout cell is not a full Bayesian treatment.
It only gives a concrete ensemble-flavored picture of why different dropout masks produce slightly different predictions.


## Exercises

1. Reduce the training set from `140` points to `80`. Which regularizer helps most?
2. Increase the dropout rate to `0.5`. At what point does optimization degrade too much?
3. Replace `AdamW` with `SGD(momentum=0.9)` and compare the impact of weight decay.
4. Plot parameter norms during training and connect the result to the weight-decay derivation.
